# 🟢 PaddleOCR LOCAL FAST V8 · CHUNKED SAFE DOWNLOAD

Corrige la detección del runtime local.

Ya **no depende de que el repo esté montado en `/content/work`**.
Detecta el entorno local por:
- Python 3.12;
- kernel WSL2;
- GPU visible.

Los archivos temporales, la caché de pip y los modelos viven bajo `/root`,
que en tu configuración Docker ya es persistente.

Versiones:
- PaddlePaddle GPU 3.2.0
- CUDA wheel cu126
- PaddleOCR 3.2.0

No usa venv y no reinicia el kernel.

Durante la instalación muestra:
- barra de descarga real;
- MB descargados / total;
- porcentaje;
- velocidad estimada por `tqdm`;
- consola de `pip`;
- heartbeat si `pip` queda silencioso.

### Cambio importante en V5

El wheel gigante de Paddle **ya no lo descarga pip**.

Se guarda persistentemente en:

`/root/.cache/paddle-wheels/`

La descarga usa `curl --continue-at -`, por lo que si falla a mitad de camino,
la próxima ejecución **reanuda** el mismo archivo en lugar de empezar desde cero.

Antes de instalar se valida que el `.whl` sea un ZIP/wheel íntegro.


### V6: progreso independiente de curl

Jupyter a veces no muestra las barras `\r` de curl.

V6 mide directamente cada 0,5 segundos cuánto crece el archivo `.whl` y muestra,
cada ~2 segundos, una línea normal como:

`812.4 MB / 1.88 GB (42.2%) · 6.4 MB/s · ETA 2.8 min`

Por eso hay feedback aunque curl permanezca totalmente silencioso.


### V7: tamaño remoto + validación en Linux

V7 ya no considera que `curl` haya terminado solo porque devolvió código 0.

1. consulta el tamaño remoto real mediante HTTP Range;
2. compara bytes locales vs remotos;
3. reanuda hasta que ambos sean exactamente iguales;
4. conserva el wheel persistente en `/root/.cache/paddle-wheels`;
5. copia el wheel completo a `/tmp`;
6. valida e instala desde `/tmp`, evitando seeks ZIP sobre el bind mount de Windows.


### V8: descarga por bloques independientes

El wheel monolítico corrupto anterior **se ignora**.

V8:
1. obtiene tamaño + CRC32 oficiales;
2. divide el wheel en bloques de 64 MiB;
3. descarga 4 bloques en paralelo con HTTP Range;
4. cada bloque solo se guarda si devuelve `206` + `Content-Range` exacto + tamaño exacto;
5. los bloques quedan persistidos bajo `/root/.cache/paddle-wheel-chunks/`;
6. si falla una ejecución, la siguiente baja únicamente bloques faltantes;
7. ensambla en `/tmp`;
8. valida CRC32 global contra BOS;
9. valida ZIP;
10. recién entonces instala.

No usa `curl -C` ni concatena descargas parciales.


In [ ]:
import sys, platform, subprocess, importlib.metadata as md, socket, pathlib

print("🟢 LOCAL FAST V3 · Paddle 3.2.0 / PaddleOCR 3.2.0")
print()

release = platform.release()
platform_text = platform.platform()
host = socket.gethostname()

print("Python:", sys.version)
print("Platform:", platform_text)
print("Kernel:", release)
print("Hostname:", host)
print("HOME:", pathlib.Path.home())
print()

# El runtime Docker local corre sobre WSL2.
is_wsl = ("microsoft" in release.lower()) or ("wsl" in release.lower())
is_py312 = sys.version_info[:2] == (3, 12)

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_text = gpu.stdout.strip()

print("GPU:", gpu_text or "No detectada")
print()

if not is_py312:
    raise RuntimeError(
        f"Este notebook espera Python 3.12 del runtime local. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

if not is_wsl:
    print("⚠️ El kernel no parece WSL2.")
    print("Esto podría ser Colab Cloud. Revisá que estés conectado al runtime local.")
    print("No voy a instalar nada automáticamente en esta celda.")
else:
    print("✅ Runtime WSL2 local detectado.")

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "NO INSTALADO"

print()
print("=== PAQUETES ===")
for name in ["paddlepaddle-gpu","paddleocr","paddlex","torch","pillow","numpy"]:
    print(f"{name:20} {ver(name)}")

In [ ]:
import sys, os, pathlib, subprocess, importlib.metadata as md, platform
import shutil, time, re, zlib, json, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

PADDLE = "3.2.0"
OCR = "3.2.0"

PADDLE_WHEEL_URL = (
    "https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/"
    "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
)

# 64 MiB por bloque: ~29 bloques para este wheel.
CHUNK_SIZE = 64 * 1024 * 1024

# Varias conexiones independientes suelen rendir mejor que una sola.
# Si tu conexión/router se pone inestable, bajalo a 2.
PARALLEL_DOWNLOADS = 4

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError(
        "ABORTADO: este kernel no parece WSL2/local. "
        "No voy a descargar Paddle por accidente en Colab Cloud."
    )

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"Este wheel requiere Python 3.12. "
        f"Encontré {sys.version_info.major}.{sys.version_info.minor}."
    )

HOME = pathlib.Path.home()
PIP_CACHE = HOME / ".cache" / "pip"
MODEL_CACHE = HOME / ".cache" / "paddlex"
CHUNK_ROOT = HOME / ".cache" / "paddle-wheel-chunks" / "3.2.0-cu126-cp312"
MANIFEST = CHUNK_ROOT / "manifest.json"

for d in (PIP_CACHE, MODEL_CACHE, CHUNK_ROOT):
    d.mkdir(parents=True, exist_ok=True)

os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)
os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODEL_CACHE)
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

WHEEL_NAME = "paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl"
TMP_WHEEL = pathlib.Path("/tmp") / WHEEL_NAME

def ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

def human_bytes(n):
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while n >= 1024 and i < len(units)-1:
        n /= 1024
        i += 1
    return f"{n:.2f} {units[i]}"

def run(cmd, **kwargs):
    print("$", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=True, **kwargs)

def remote_metadata(url):
    """
    Pide 1 byte para obtener:
      - Content-Range => tamaño total
      - x-bce-content-crc32 => CRC32 oficial
      - ETag => identidad del objeto
    """
    curl = shutil.which("curl")
    p = subprocess.run(
        [
            curl,
            "--location",
            "--silent",
            "--show-error",
            "--fail",
            "--range", "0-0",
            "--dump-header", "-",
            "--output", "/dev/null",
            url,
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    headers = p.stdout

    m_total = re.findall(
        r"(?im)^content-range:\s*bytes\s+\d+-\d+/(\d+)\s*$",
        headers,
    )
    m_crc = re.findall(
        r"(?im)^x-bce-content-crc32:\s*(\d+)\s*$",
        headers,
    )
    m_etag = re.findall(
        r'(?im)^etag:\s*"?([^"\r\n]+)"?\s*$',
        headers,
    )

    if not m_total:
        raise RuntimeError("El CDN no informó Content-Range/tamaño total.")
    if not m_crc:
        raise RuntimeError("El CDN no informó x-bce-content-crc32.")

    return {
        "total": int(m_total[-1]),
        "crc32": int(m_crc[-1]),
        "etag": m_etag[-1] if m_etag else None,
    }

def build_chunks(total):
    chunks = []
    start = 0
    idx = 0
    while start < total:
        end = min(total - 1, start + CHUNK_SIZE - 1)
        chunks.append((idx, start, end))
        idx += 1
        start = end + 1
    return chunks

def chunk_path(idx, start, end):
    return CHUNK_ROOT / f"{idx:04d}_{start:012d}-{end:012d}.bin"

def chunk_is_complete(idx, start, end):
    p = chunk_path(idx, start, end)
    return p.exists() and p.stat().st_size == (end - start + 1)

def download_one_chunk(item):
    """
    Descarga un Range a un .tmp y SOLO lo renombra a .bin si:
      - curl termina 0
      - HTTP 206
      - Content-Range coincide EXACTAMENTE
      - tamaño coincide EXACTAMENTE

    Nunca concatena/reanuda un archivo monolítico.
    """
    idx, start, end = item
    final = chunk_path(idx, start, end)
    expected = end - start + 1

    if final.exists() and final.stat().st_size == expected:
        return idx, expected, "cached"

    tmp = final.with_suffix(".tmp")
    headers = final.with_suffix(".headers")

    for p in (tmp, headers):
        if p.exists():
            p.unlink()

    curl = shutil.which("curl")

    cmd = [
        curl,
        "--location",
        "--fail",
        "--silent",
        "--show-error",
        "--retry", "10",
        "--retry-delay", "1",
        "--retry-all-errors",
        "--connect-timeout", "30",
        "--range", f"{start}-{end}",
        "--dump-header", str(headers),
        "--output", str(tmp),
        PADDLE_WHEEL_URL,
    ]

    p = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )

    hdr = headers.read_text(errors="replace") if headers.exists() else ""
    got_size = tmp.stat().st_size if tmp.exists() else 0

    statuses = re.findall(r"HTTP/\S+\s+(\d+)", hdr)
    ranges = re.findall(
        r"(?im)^content-range:\s*bytes\s+(\d+)-(\d+)/(\d+)\s*$",
        hdr,
    )

    valid_range = bool(
        ranges
        and int(ranges[-1][0]) == start
        and int(ranges[-1][1]) == end
    )
    valid_status = "206" in statuses

    if (
        p.returncode != 0
        or not valid_status
        or not valid_range
        or got_size != expected
    ):
        detail = {
            "chunk": idx,
            "range": f"{start}-{end}",
            "returncode": p.returncode,
            "statuses": statuses,
            "content_ranges": ranges,
            "expected_size": expected,
            "got_size": got_size,
            "stderr": p.stderr[-1000:],
        }
        # El .tmp no se considera cache válido.
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(json.dumps(detail, ensure_ascii=False))

    # Rename atómico: un .bin presente significa "bloque validado".
    tmp.rename(final)
    if headers.exists():
        headers.unlink()

    return idx, expected, "downloaded"

def download_all_chunks(chunks):
    cached_bytes = sum(
        (end-start+1)
        for idx, start, end in chunks
        if chunk_is_complete(idx, start, end)
    )
    missing = [
        item for item in chunks
        if not chunk_is_complete(*item)
    ]

    total = sum(end-start+1 for _, start, end in chunks)

    print()
    print("=== CACHE DE BLOQUES ===")
    print("Bloques totales:      ", len(chunks))
    print("Bloques ya completos: ", len(chunks) - len(missing))
    print("Bloques por descargar:", len(missing))
    print("Datos ya cacheados:   ", human_bytes(cached_bytes))
    print("Total wheel:          ", human_bytes(total))
    print("Conexiones paralelas: ", PARALLEL_DOWNLOADS)
    print()

    if not missing:
        print("✅ Todos los bloques ya están en cache.")
        return

    lock = threading.Lock()
    done_bytes = cached_bytes
    started = time.time()

    bar = tqdm(
        total=total,
        initial=cached_bytes,
        desc="Paddle wheel",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        dynamic_ncols=True,
        leave=True,
    )

    errors = []

    with ThreadPoolExecutor(max_workers=PARALLEL_DOWNLOADS) as pool:
        future_map = {
            pool.submit(download_one_chunk, item): item
            for item in missing
        }

        for fut in as_completed(future_map):
            item = future_map[fut]
            idx, start, end = item
            expected = end-start+1

            try:
                _, size, mode = fut.result()
                with lock:
                    done_bytes += size
                    bar.update(size)

                    elapsed = max(0.001, time.time()-started)
                    net_new = max(0, done_bytes-cached_bytes)
                    speed = net_new/elapsed
                    remaining = total-done_bytes
                    eta = remaining/speed if speed > 0 else None

                    eta_txt = (
                        f"{eta/60:.1f} min"
                        if eta is not None and eta >= 60
                        else f"{eta:.0f} s"
                        if eta is not None
                        else "calculando"
                    )

                    bar.set_postfix_str(
                        f"chunk {idx+1}/{len(chunks)} · "
                        f"{human_bytes(speed)}/s · ETA {eta_txt}"
                    )

                    print(
                        f"✅ bloque {idx+1:02d}/{len(chunks)} "
                        f"{human_bytes(size)} · "
                        f"total {done_bytes/total*100:5.1f}%",
                        flush=True,
                    )

            except Exception as e:
                errors.append((item, str(e)))
                print(
                    f"❌ bloque {idx+1}/{len(chunks)} falló: {e}",
                    flush=True,
                )

    bar.close()

    if errors:
        print()
        print("⚠️ Algunos bloques fallaron.")
        print("Los bloques buenos quedaron guardados.")
        print("Volvé a ejecutar esta celda: SOLO se descargarán los faltantes.")
        for item, err in errors[:5]:
            print(" -", item, err[:500])
        raise RuntimeError(
            f"{len(errors)} bloque(s) fallaron; cache parcial conservada."
        )

    print()
    print("✅ Todos los bloques fueron descargados con Range 206 válido.")

def assemble_and_crc(chunks, expected_total, expected_crc):
    """
    Ensambla secuencialmente en /tmp (filesystem Linux) y calcula CRC32
    en la MISMA pasada. No hace seeks sobre el bind mount persistente.
    """
    if TMP_WHEEL.exists():
        TMP_WHEEL.unlink()

    crc = 0
    written = 0

    print()
    print("=== ENSAMBLANDO + CRC32 ===")

    with TMP_WHEEL.open("wb") as out, tqdm(
        total=expected_total,
        desc="Ensamblando",
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        dynamic_ncols=True,
        leave=True,
    ) as bar:
        for idx, start, end in chunks:
            p = chunk_path(idx, start, end)
            expected = end-start+1

            if not p.exists() or p.stat().st_size != expected:
                raise RuntimeError(f"Falta/incompleto bloque {idx}: {p}")

            with p.open("rb") as inp:
                while True:
                    data = inp.read(16 * 1024 * 1024)
                    if not data:
                        break
                    out.write(data)
                    crc = zlib.crc32(data, crc)
                    written += len(data)
                    bar.update(len(data))

    crc &= 0xFFFFFFFF

    print()
    print("Tamaño ensamblado:", written)
    print("Tamaño esperado:  ", expected_total)
    print("CRC32 local:       ", crc)
    print("CRC32 oficial:     ", expected_crc)

    if written != expected_total:
        raise RuntimeError("El tamaño ensamblado no coincide.")

    if crc != expected_crc:
        raise RuntimeError(
            "CRC32 final NO coincide. "
            "No voy a instalar este wheel."
        )

    print("✅ CRC32 coincide con el objeto oficial.")
    return crc

def validate_zip(path):
    print()
    print("=== VALIDANDO ZIP/WHEEL ===")
    unzip = shutil.which("unzip")

    if unzip:
        p = subprocess.run(
            [unzip, "-tqq", str(path)],
            capture_output=True,
            text=True,
        )
        if p.returncode != 0:
            raise RuntimeError(
                "unzip -t falló:\n"
                + (p.stdout + "\n" + p.stderr)[-2000:]
            )
        print("✅ unzip -t: OK")
        return

    p = subprocess.run(
        [sys.executable, "-m", "zipfile", "-t", str(path)],
        capture_output=True,
        text=True,
    )
    if p.returncode != 0:
        raise RuntimeError(
            "python -m zipfile -t falló:\n"
            + (p.stdout + "\n" + p.stderr)[-2000:]
        )
    print("✅ zipfile -t: OK")

print("🟢 LOCAL FAST V8 · CHUNKED SAFE DOWNLOAD")
print()
print("Cache pip:    ", PIP_CACHE)
print("Cache modelos:", MODEL_CACHE)
print("Cache bloques:", CHUNK_ROOT)
print()

if ver("paddlepaddle-gpu") == PADDLE:
    print("✅ PaddlePaddle GPU 3.2.0 ya está instalado.")
else:
    existing = ver("paddlepaddle-gpu")
    if existing:
        raise RuntimeError(
            f"Hay PaddlePaddle GPU {existing} instalado. "
            "No voy a mezclar versiones."
        )

    print("🌐 Leyendo metadata oficial...")
    meta = remote_metadata(PADDLE_WHEEL_URL)

    print("Tamaño oficial:", human_bytes(meta["total"]), meta["total"])
    print("CRC32 oficial: ", meta["crc32"])
    print("ETag:           ", meta["etag"])

    # Guardar metadata para trazabilidad.
    MANIFEST.write_text(
        json.dumps(meta, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    chunks = build_chunks(meta["total"])

    download_all_chunks(chunks)
    assemble_and_crc(
        chunks,
        expected_total=meta["total"],
        expected_crc=meta["crc32"],
    )
    validate_zip(TMP_WHEEL)

    print()
    print("=== INSTALANDO PADDLE DESDE WHEEL VERIFICADO ===")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "180",
        str(TMP_WHEEL),
    ])

if ver("paddleocr") == OCR:
    print("✅ PaddleOCR 3.2.0 ya está instalado.")
else:
    existing = ver("paddleocr")
    if existing:
        raise RuntimeError(
            f"Hay PaddleOCR {existing} instalado. "
            "No voy a mezclar versiones."
        )

    print()
    print("=== INSTALANDO PADDLEOCR ===")
    run([
        sys.executable, "-m", "pip", "install",
        "--retries", "10",
        "--timeout", "180",
        f"paddleocr=={OCR}",
    ])

print()
print("✅ Instalación lista.")
print("La siguiente celda hace el smoke test en un proceso Python nuevo.")

In [ ]:
import sys, subprocess, pathlib, os, platform

release = platform.release().lower()
if "microsoft" not in release and "wsl" not in release:
    raise RuntimeError("No parece el runtime local WSL2.")

dev_dir = pathlib.Path.home()/".cache"/"paddleocr-dev"
dev_dir.mkdir(parents=True, exist_ok=True)

worker_path = dev_dir/"paddle_smoke_worker_v3.py"
worker_path.write_text('\nimport os, sys, json\nfrom pathlib import Path\n\nos.environ.setdefault("PADDLE_PDX_MODEL_SOURCE", "BOS")\nos.environ.setdefault("PADDLE_PDX_CACHE_HOME", str(Path.home()/".cache"/"paddlex"))\n\nprint("=== WORKER NUEVO ===", flush=True)\nprint("Python:", sys.version, flush=True)\n\nimport paddle\nprint("Paddle:", paddle.__version__, flush=True)\nprint("CUDA:", paddle.is_compiled_with_cuda(), flush=True)\nprint("GPU count:", paddle.device.cuda.device_count(), flush=True)\n\nif not paddle.is_compiled_with_cuda() or paddle.device.cuda.device_count() < 1:\n    raise RuntimeError("Paddle no ve la GPU CUDA.")\n\npaddle.set_device("gpu:0")\ntry:\n    print("GPU:", paddle.device.cuda.get_device_name(), flush=True)\nexcept Exception:\n    print("GPU: gpu:0", flush=True)\n\nimport PIL\nfrom PIL import Image, ImageDraw\nprint("Pillow:", PIL.__version__, flush=True)\n\nfrom paddleocr import PaddleOCR\nimport paddleocr\nprint("PaddleOCR:", getattr(paddleocr, "__version__", "unknown"), flush=True)\n\ndev_dir = Path.home()/".cache"/"paddleocr-dev"\ndev_dir.mkdir(parents=True, exist_ok=True)\n\nimg_path = dev_dir/"paddle_smoke_es.png"\nimg = Image.new("RGB", (1400, 320), "white")\nImageDraw.Draw(img).text(\n    (50, 100),\n    "Histologia epitelio plano simple prueba OCR espanol 12345",\n    fill="black"\n)\nimg.save(img_path)\n\nprint("Imagen:", img_path, flush=True)\nprint("Inicializando OCR...", flush=True)\n\nocr = PaddleOCR(\n    lang="es",\n    device="gpu:0",\n    use_doc_orientation_classify=False,\n    use_doc_unwarping=False,\n    use_textline_orientation=False,\n)\n\nprint("Ejecutando OCR...", flush=True)\nresults = ocr.predict(str(img_path))\n\ntexts = []\nfor res in results:\n    d = getattr(res, "json", res)\n    if callable(d):\n        d = d()\n    if isinstance(d, dict) and "res" in d:\n        d = d["res"]\n    if isinstance(d, dict):\n        texts += [str(x) for x in d.get("rec_texts", [])]\n\nprint("Textos:", json.dumps(texts, ensure_ascii=False), flush=True)\n\nif not texts:\n    raise RuntimeError("No se reconoció texto.")\n\nprint("✅ SMOKE TEST COMPLETO", flush=True)\n', encoding="utf-8")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
env["PADDLE_PDX_CACHE_HOME"] = str(pathlib.Path.home()/".cache"/"paddlex")

print("Ejecutando worker fresco:")
print(worker_path)
print()

proc = subprocess.Popen(
    [sys.executable, str(worker_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc:
    raise RuntimeError(f"Worker falló con código {rc}")